# Алмазов МРТ — финальная оценка модели, выбранной по валидации
Запускать **после** `Almazov_MRI_experiments.ipynb`: T4 GPU → «Выполнить все» (~7 минут, обучения нет).

Берёт модель, которую эксперименты выбрали по валидации (`experiments_v1.json → selected`), и считает для неё **все те же метрики, что в основной работе**:
доверительные интервалы (бутстрэп по группам), уникальные снимки, новые пациенты Figshare, разрешение, режим скрининга, матрица ошибок.
Тест для выбора модели не используется — здесь он только измеряет уже выбранную модель.
Результат: `almazov/results_final.json`, `almazov/figs/cm_final.png`, веса для демо в `almazov/final/`.

In [ ]:
# === 0. НАСТРОЙКА ===
!nvidia-smi -L
!pip -q install timm segmentation-models-pytorch h5py
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, re, random, hashlib, glob, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
import matplotlib.pyplot as plt
cv2.setNumThreads(0)
W = Path('/content/drive/MyDrive/almazov'); W.mkdir(parents=True, exist_ok=True)
(W/'figs').mkdir(exist_ok=True); (W/'ckpt').mkdir(exist_ok=True)
ORG = Path('/content/org'); FIG = Path('/content/fig')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'Нет GPU: Среда выполнения -> Сменить среду -> T4 GPU'
CLASSES = ['Глиома', 'Менингиома', 'Гипофиз', 'Опухоли нет']   # индекс 3 = нет опухоли
print('OK, папка результатов:', W)

In [ ]:
# === 1. ДАННЫЕ ОРГАНИЗАТОРОВ: распаковка Almazov_MRI.zip из Google Drive ===
# Набор зафиксирован: 9150 из 12064 файлов (остальные источник не отдал). Пропуски распределены по всем классам.
def n_imgs(root): return sum(1 for p in Path(root).rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
if not ORG.exists() or n_imgs(ORG) < 9000:
    zs = sorted(glob.glob('/content/drive/MyDrive/**/Almazov_MRI*.zip', recursive=True))
    assert zs, 'Не нашёл Almazov_MRI.zip в Google Drive'
    z = zipfile.ZipFile(zs[0]); cnt = 0
    for info in z.infolist():
        name = info.filename
        if not (info.flag_bits & 0x800):          # zip из Windows без UTF-8: чиним кириллицу
            try: name = name.encode('cp437').decode('cp866')
            except Exception: pass
        parts = name.replace('\\', '/').split('/')
        if parts and parts[0].startswith('Almazov_MRI'): parts = parts[1:]
        if info.is_dir() or not parts or not parts[-1].lower().endswith(('.jpg', '.jpeg', '.png')): continue
        dst = ORG.joinpath(*parts); dst.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as s, open(dst, 'wb') as d: shutil.copyfileobj(s, d)
        cnt += 1
    print('Распаковано из', zs[0], ':', cnt)
print('Снимков локально:', n_imgs(ORG))

In [ ]:
# === 2. FIGSHARE (Cheng): 3064 снимка T1+контраст, маски опухолей от радиологов, ID пациентов ===
import h5py
FIGNPZ = W/'figshare256.npz'
if not FIGNPZ.exists():
    FIG.mkdir(exist_ok=True)
    for fid in [3381290, 3381293, 3381296, 3381302]:
        z = FIG/f'{fid}.zip'
        if not z.exists():
            !wget -q -O "{z}" https://ndownloader.figshare.com/files/{fid}
        zipfile.ZipFile(z).extractall(FIG/'mat')
    files = sorted(glob.glob(str(FIG/'mat'/'**'/'*.mat'), recursive=True), key=lambda s: int(Path(s).stem))
    X, M, Y, P, IDX = [], [], [], [], []
    for f in files:
        with h5py.File(f, 'r') as h:
            cj = h['cjdata']
            img = np.array(cj['image']).T.astype(np.float32)
            msk = np.array(cj['tumorMask']).T.astype(np.uint8)
            lab = int(np.array(cj['label']).flatten()[0])
            pid = ''.join(chr(int(c)) for c in np.array(cj['PID']).flatten())
        lo, hi = np.percentile(img, 0.5), np.percentile(img, 99.5)
        img = np.clip((img-lo)/(hi-lo+1e-6), 0, 1)*255
        X.append(cv2.resize(img.astype(np.uint8), (256, 256), interpolation=cv2.INTER_AREA))
        M.append(cv2.resize(msk, (256, 256), interpolation=cv2.INTER_NEAREST))
        Y.append(lab); P.append(pid); IDX.append(int(Path(f).stem))
    cheng2ours = {1: 1, 2: 0, 3: 2}     # Cheng: 1 менингиома, 2 глиома, 3 гипофиз
    np.savez_compressed(FIGNPZ, X=np.stack(X), M=np.stack(M), y=np.array([cheng2ours[v] for v in Y]),
                        pid=np.array(P), idx=np.array(IDX))
F = np.load(FIGNPZ)
fX, fM, fy, fpid = F['X'], F['M'], F['y'], F['pid'].astype(str)
print('Figshare:', fX.shape, 'пациентов:', len(set(fpid)), 'по классам:', np.bincount(fy))

In [ ]:
# === 3. ПРЕДОБРАБОТКА + АУДИТ ДАННЫХ (дубли, аугментированные копии, общие пациенты с Figshare) ===
from PIL import Image
def prep(g):
    # единая предобработка для любых источников: серый -> квадрат с чёрными полями -> 256x256 -> контраст по перцентилям
    h, w = g.shape; s = max(h, w)
    if h != w:
        c = np.zeros((s, s), g.dtype); c[(s-h)//2:(s-h)//2+h, (s-w)//2:(s-w)//2+w] = g; g = c
    g = cv2.resize(g, (256, 256), interpolation=cv2.INTER_AREA if s > 256 else cv2.INTER_LINEAR).astype(np.float32)
    lo, hi = np.percentile(g, 0.5), np.percentile(g, 99.5)
    return (np.clip((g - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype(np.uint8)

files = sorted(p for p in ORG.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
rows, imgs = [], []
for p in files:
    parts = [s.strip() for s in p.relative_to(ORG).parts]
    split = 'test' if any(s.startswith('Тест') for s in parts) else 'train'
    lab = next(s for s in parts if s in CLASSES)
    g = np.array(Image.open(p).convert('L'))
    imgs.append(prep(g))
    rows.append(dict(path=str(p), name=p.name, split=split, label=CLASSES.index(lab), h=g.shape[0], w=g.shape[1],
                     pix=hashlib.sha1(g.tobytes()).hexdigest()))
df = pd.DataFrame(rows); A = np.stack(imgs); n = len(df); LAB = df.label.values
def source(nm):
    if re.match(r'^T[re]-', nm): return 'Kaggle Tr/Te'
    if re.search(r'_(HF|VF|RO|BR|SP|DA)_\.', nm): return 'аугм. копии'
    if re.match(r'^[PGMN]_\d+\.', nm): return 'P/G/M/N_n'
    if re.match(r'^\d+\.', nm): return 'числовые'
    if re.match(r'^(gg|m|p|image)', nm, re.I): return 'SARTAJ-подобные'
    return 'Br35H/прочие'
df['source'] = df.name.map(source)
df['augbase'] = df.name.str.extract(r'^([PGMN]_\d+)(?:_[A-Z]{2}_)?\.', expand=False)
print(df.groupby(['split', 'label']).size().unstack().rename(columns=dict(enumerate(CLASSES))))
print(pd.crosstab(df.source, df.split))

# --- похожесть снимков: корреляция 64x64 с учётом 8 поворотов/отражений ---
def small(arr, s=64): return np.stack([cv2.resize(a, (s, s), interpolation=cv2.INTER_AREA) for a in arr]).astype(np.float32)
def zt(x):
    x = x.reshape(len(x), -1); x = x - x.mean(1, keepdim=True); return x / (x.norm(dim=1, keepdim=True) + 1e-6)
def d8_t(x, t):
    y = torch.rot90(x, t // 2, (1, 2)); return torch.flip(y, (2,)) if t % 2 else y
S64 = torch.tensor(small(A)); Z = zt(S64).to(dev); ZD = [zt(d8_t(S64, t)).to(dev) for t in range(8)]
I, J, C = [], [], []
for s in range(0, n, 512):
    e = min(n, s + 512)
    best = torch.stack([ZD[t][s:e] @ Z.T for t in range(8)]).max(0).values
    best[torch.arange(e - s, device=dev), torch.arange(s, e, device=dev)] = 0
    ii, jj = torch.nonzero(best >= 0.97, as_tuple=True)
    keep = (ii + s) < jj
    I.append((ii[keep] + s).cpu().numpy()); J.append(jj[keep].cpu().numpy()); C.append(best[ii[keep], jj[keep]].cpu().numpy())
I, J, C = np.concatenate(I), np.concatenate(J), np.concatenate(C)
FZt = zt(torch.tensor(small(fX))).to(dev)
fbest = np.zeros(n, np.float32); farg = np.zeros(n, int)
for s in range(0, n, 512):
    e = min(n, s + 512)
    c = torch.stack([ZD[t][s:e] @ FZt.T for t in range(8)]).max(0).values
    v, a = c.max(1); fbest[s:e] = v.cpu().numpy(); farg[s:e] = a.cpu().numpy()
fig_ok = (LAB != 3) & (fy[farg] == LAB)
del Z, ZD, FZt; torch.cuda.empty_cache()

def build(thr_dup, thr_fig, use_pid, use_names):
    par = list(range(n))
    def find(a):
        while par[a] != a:
            par[a] = par[par[a]]; a = par[a]
        return a
    def un(a, b):
        a, b = find(a), find(b)
        if a != b: par[a] = b
    for ids in df.groupby('pix').groups.values():
        ids = list(ids)
        for i in ids[1:]: un(ids[0], i)
    if use_names:
        for ids in df.dropna(subset=['augbase']).groupby(['augbase', 'label']).groups.values():
            ids = list(ids)
            for i in ids[1:]: un(ids[0], i)
    m = (C >= thr_dup) & (LAB[I] == LAB[J])
    for i, j in zip(I[m].tolist(), J[m].tolist()): un(i, j)
    pid = np.where((fbest >= thr_fig) & fig_ok, fpid[farg], '')
    if use_pid:
        first = {}
        for i in np.where(pid != '')[0].tolist():
            if pid[i] in first: un(first[pid[i]], i)
            else: first[pid[i]] = i
    return np.array([find(i) for i in range(n)]), pid

te = (df.split == 'test').values
def leak(g):
    trg = set(g[~te]); return te & np.isin(g, list(trg))
print('\nЧувствительность к порогу похожести:')
for thr in (0.97, 0.98, 0.99):
    gi, _ = build(thr, thr, False, False); gs, _ = build(thr, thr, True, False)
    print(f'  порог {thr}: тест-снимков с копией в обучении {leak(gi).sum()}/{te.sum()},'
          f' с копией или тем же пациентом {leak(gs).sum()}/{te.sum()}')
THR = 0.98
df['group'], pid = build(THR, THR, True, True)       # для разбиения: самый осторожный вариант
df['imgcomp'], _ = build(THR, THR, False, False)     # «тот же снимок» (копия, отражение, ресайз)
gstat, _ = build(THR, THR, True, False)
df['pid'] = pid; df['fig_i'] = np.where(pid != '', farg, -1)
df['leak_copy'] = leak(df.imgcomp.values); df['leak_patient'] = leak(gstat)
df['label_conflict'] = df.group.map(df.groupby('group').label.nunique() > 1).values
audit = dict(files=n, groups=int(df.group.nunique()), biggest_group=int(df.group.value_counts().max()),
             exact_dup_groups=int((df.groupby('pix').size() > 1).sum()),
             org_images_matched_to_figshare=int((pid != '').sum()), figshare_patients_found=len(set(pid) - {''}),
             test_images=int(te.sum()), test_with_copy_in_train=int(df.leak_copy.sum()),
             test_with_copy_or_same_patient_in_train=int(df.leak_patient.sum()),
             label_conflict_images=int(df.label_conflict.sum()))
print(json.dumps(audit, ensure_ascii=False, indent=1))
print(df[te].groupby('label')[['leak_copy', 'leak_patient']].mean().rename(index=dict(enumerate(CLASSES))).round(3))
# (аудит уже сохранён основным блокнотом; здесь не перезаписываем)

In [ ]:
# === 5. ЧЕСТНОЕ РАЗБИЕНИЕ: по группам «пациент / снимок», а не по картинкам ===
from sklearn.model_selection import StratifiedGroupKFold
SPLIT = W/'split_v3.csv'
if SPLIT.exists():
    sv = pd.read_csv(SPLIT, keep_default_na=False).set_index('path')
    assert set(sv.index) == set(df.path), 'Набор файлов изменился: удали split_v3.csv в Drive'
    for col in ['part', 'test_unique', 'group', 'imgcomp', 'pid', 'fig_i']:
        df[col] = df.path.map(sv[col]).values
    df['test_unique'] = df.test_unique.astype(str).eq('True'); df['pid'] = df.pid.astype(str); df['fig_i'] = df.fig_i.astype(int)
    print('Разбиение загружено из Drive (то же, что раньше).')
else:
    raise SystemExit('Нет split_v3.csv в Drive (MyDrive/almazov). Сначала нужен основной блокнот: эксперименты должны идти на том же разбиении.')
    idx = np.arange(n)
    rest, test = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(idx, LAB, df.group.values))
    tr_i, va_i = next(StratifiedGroupKFold(6, shuffle=True, random_state=SEED).split(rest, LAB[rest], df.group.values[rest]))
    part = np.array(['train'] * n, dtype=object); part[test] = 'test'; part[rest[va_i]] = 'val'
    df['part'] = part
    tm = (df.part == 'test') & (df.source != 'аугм. копии')
    df['test_unique'] = False
    df.loc[df[tm].drop_duplicates('imgcomp').index, 'test_unique'] = True
    df[['path', 'name', 'split', 'label', 'source', 'group', 'imgcomp', 'pid', 'fig_i', 'part', 'test_unique',
        'leak_copy', 'leak_patient']].to_csv(SPLIT, index=False)
    print('Разбиение создано и сохранено в Drive.')
for a_, b_ in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not (set(df.group[df.part == a_]) & set(df.group[df.part == b_])), (a_, b_)
use = ~df.label_conflict.values
TR = np.where((df.part == 'train').values & use)[0]; VA = np.where((df.part == 'val').values & use)[0]
TE = np.where((df.part == 'test').values & use)[0]; TEU = np.where(df.test_unique.values & use)[0]
print(pd.crosstab(df.part, df.label).rename(columns=dict(enumerate(CLASSES))))
print('Уникальных снимков в тесте (без копий и аугментаций):', len(TEU))
# разбиение Figshare для сегментации — по тем же пациентам
pid_part = {}
for p_, pt in zip(df.pid, df.part):
    if p_: assert pid_part.setdefault(p_, pt) == pt
fpart = np.array([pid_part.get(p_, 'train') for p_ in fpid])
s_tr, s_va, s_te = (np.where(fpart == k)[0] for k in ('train', 'val', 'test'))
print('Figshare пациентов train/val/test:', len(set(fpid[s_tr])), len(set(fpid[s_va])), len(set(fpid[s_te])),
      '| снимков:', len(s_tr), len(s_va), len(s_te))

In [ ]:
# === E1. ОБЩИЕ ФУНКЦИИ: обучение классификатора с параметрами эксперимента ===
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, roc_auc_score
MEAN, STD = 0.449*255, 0.226*255
EXP = W/'experiments_v1.json'
R = json.load(open(EXP)) if EXP.exists() else {}
def save_R(): json.dump(R, open(EXP, 'w'), ensure_ascii=False, indent=1)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class CDS(Dataset):
    # тот же датасет, что в основном блокноте, + параметры: аугментации вкл/выкл и размер входа
    def __init__(s, idx, aug, size=256): s.idx, s.aug, s.size = np.asarray(idx), aug, size
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = s.idx[k]; x = A[i].astype(np.float32)
        if s.aug:
            if random.random() < .5: x = x[:, ::-1]
            M_ = cv2.getRotationMatrix2D((128, 128), random.uniform(-15, 15), random.uniform(.9, 1.1))
            M_[:, 2] += np.random.uniform(-12, 12, 2)
            x = cv2.warpAffine(np.ascontiguousarray(x), M_, (256, 256), borderMode=cv2.BORDER_CONSTANT)
            x = x * random.uniform(.8, 1.2) + random.uniform(-20, 20)
            if random.random() < .3: x = cv2.GaussianBlur(x, (3, 3), 0)
            if random.random() < .3: x = x + np.random.normal(0, 5, x.shape)
        if s.size != 256: x = cv2.resize(np.ascontiguousarray(x), (s.size, s.size), interpolation=cv2.INTER_AREA)
        x = (np.clip(x, 0, 255) - MEAN) / STD
        return torch.tensor(x, dtype=torch.float32)[None].repeat(3, 1, 1), int(LAB[i])

@torch.no_grad()
def cls_predict(m, idx, size=256, bs=128):
    m.eval(); P = []
    for x, _ in DataLoader(CDS(idx, False, size), bs, shuffle=False, num_workers=2, pin_memory=True):
        x = x.to(dev)
        with torch.autocast('cuda'):
            p = m(x).float().softmax(1) + m(torch.flip(x, (3,))).float().softmax(1)
        P.append((p / 2).cpu())
    return torch.cat(P).numpy()

def evaluate(P, Y):
    pr = P.argmax(1); tt = Y != 3; pt = 1 - P[:, 3]
    return dict(n=int(len(Y)), accuracy=float(accuracy_score(Y, pr)), macro_f1=float(f1_score(Y, pr, average='macro')),
                balanced_acc=float(balanced_accuracy_score(Y, pr)),
                glioma_recall=float((pr[Y == 0] == 0).mean()),
                tumor_sens=float(((pr != 3) & tt).sum() / max(tt.sum(), 1)),
                tumor_spec=float(((pr == 3) & ~tt).sum() / max((~tt).sum(), 1)),
                screen_sens_05=float(((pt >= .5) & tt).sum() / max(tt.sum(), 1)),
                screen_spec_05=float(((pt < .5) & ~tt).sum() / max((~tt).sum(), 1)),
                tumor_auc=float(roc_auc_score(tt, pt)))

PROB = W/'exp_probs'; PROB.mkdir(exist_ok=True)
def keep_probs(tag, Pva, Pte): np.savez(PROB/f'{tag}.npz', va=Pva, te=Pte)   # вероятности — для ансамбля без переобучения
def run_cls(tag, what, arch='efficientnet_b0', pretrained=True, aug=True, size=256, seed=42, epochs=15, lr=3e-4, bs=64, cw=False):
    if tag in R: print(f'[{tag}] уже готов: accuracy {R[tag]["test"]["accuracy"]:.4f}'); return
    set_seed(seed)
    m = timm.create_model(arch, pretrained=pretrained, num_classes=4).to(dev)
    dl = DataLoader(CDS(TR, aug, size), bs, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
    opt = torch.optim.AdamW(m.parameters(), lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=epochs * len(dl), pct_start=.15)
    scaler = torch.cuda.amp.GradScaler()
    wts = None
    if cw:   # веса классов обратно пропорциональны частоте в обучении (редкие/трудные классы весят больше)
        cnt = np.bincount(LAB[TR], minlength=4).astype(np.float32); wts = torch.tensor(cnt.sum() / (4 * cnt), device=dev)
        print(f'[{tag}] веса классов:', dict(zip(CLASSES, np.round(wts.cpu().numpy(), 2))))
    crit = nn.CrossEntropyLoss(weight=wts, label_smoothing=.05)
    CK, BEST = W/'ckpt'/f'exp_{tag}_last.pt', W/'ckpt'/f'exp_{tag}_best.pt'
    start, best, best_ep = 0, -1.0, -1
    if CK.exists():
        c = torch.load(CK, map_location=dev, weights_only=False)
        m.load_state_dict(c['m']); opt.load_state_dict(c['o']); sch.load_state_dict(c['s']); scaler.load_state_dict(c['sc'])
        start, best, best_ep = c['ep'] + 1, c['best'], c.get('best_ep', -1); print(f'[{tag}] продолжаю с эпохи', start + 1)
    t_all = time.time()
    for ep in range(start, epochs):
        m.train(); t0 = time.time(); tl = 0
        for x, y in dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda'): loss = crit(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += loss.item()
        f1 = f1_score(LAB[VA], cls_predict(m, VA, size).argmax(1), average='macro')
        if f1 > best: best, best_ep = f1, ep; torch.save(m.state_dict(), BEST)
        torch.save(dict(m=m.state_dict(), o=opt.state_dict(), s=sch.state_dict(), sc=scaler.state_dict(), ep=ep, best=best, best_ep=best_ep), CK)
        print(f'[{tag}] эпоха {ep+1}/{epochs}: loss {tl/len(dl):.3f}  val macroF1 {f1:.4f}  лучший {best:.4f}  {time.time()-t0:.0f}с')
    m.load_state_dict(torch.load(BEST, map_location=dev))
    P = cls_predict(m, TE, size); keep_probs(tag, cls_predict(m, VA, size), P)
    R[tag] = dict(what=what, arch=arch, pretrained=pretrained, aug=aug, size=size, seed=seed, epochs=epochs, class_weights=cw,
                  params_M=round(sum(p.numel() for p in m.parameters()) / 1e6, 1), val_macro_f1=float(best), best_epoch=int(best_ep) + 1,
                  test=evaluate(P, LAB[TE]))
    save_R(); CK.unlink(missing_ok=True)          # промежуточный чекпоинт больше не нужен (экономим место в Drive)
    t = R[tag]['test']; print(f'[{tag}] ТЕСТ: accuracy {t["accuracy"]:.4f}  macroF1 {t["macro_f1"]:.4f}  чувств. {t["tumor_sens"]:.4f}  спец. {t["tumor_spec"]:.4f}')
    del m, opt; torch.cuda.empty_cache()

# основная модель (уже обучена в основном блокноте) — та же оценка тем же кодом, для сверки
if 'base' not in R:
    mb = timm.create_model('efficientnet_b0', pretrained=False, num_classes=4).to(dev)
    mb.load_state_dict(torch.load(W/'ckpt'/'cls_v3_best.pt', map_location=dev))
    Pva_b, Pte_b = cls_predict(mb, VA), cls_predict(mb, TE); keep_probs('base', Pva_b, Pte_b)
    R['base'] = dict(what='Основная модель (EfficientNet-B0, ImageNet, аугментации, 256 px)', arch='efficientnet_b0', pretrained=True,
                     aug=True, size=256, seed=42, epochs=15, val_macro_f1=float(f1_score(LAB[VA], Pva_b.argmax(1), average='macro')),
                     test=evaluate(Pte_b, LAB[TE]))
    save_R(); del mb; torch.cuda.empty_cache()
if (W/'results_cls_v3.json').exists():   # сверка: должно совпасть с основным блокнотом
    ref = json.load(open(W/'results_cls_v3.json'))['test_group']['accuracy']
    print(f'Сверка с основным блокнотом: accuracy {R["base"]["test"]["accuracy"]:.4f} (было {ref:.4f})',
          'OK' if abs(R['base']['test']['accuracy'] - ref) < 0.003 else '<- НЕ СОВПАДАЕТ, напиши мне')
def safe(fn, *a, **k):
    # если один эксперимент упал (например, не хватило памяти) — остальные всё равно выполнятся
    try: fn(*a, **k)
    except Exception as e:
        print(f'!!! эксперимент {a[0]} упал: {type(e).__name__}: {e}'); torch.cuda.empty_cache()
print('Основная модель на честном тесте:', {k: round(v, 4) for k, v in R['base']['test'].items() if k != 'n'})

In [ ]:
# === F0. УСТОЙЧИВОСТЬ ВЫБРАННОЙ АРХИТЕКТУРЫ: ещё 2 обучения того же рецепта с другим seed (~20 мин) ===
# Выбор модели НЕ меняется (он уже сделан по валидации) — повторы только показывают, что результат не случайность.
sel0 = R['selected']['tag']
if sel0 in R and 'arch' in R[sel0] and not R[sel0].get('members'):
    a0 = R[sel0]
    for sd in (1, 2):
        safe(run_cls, f'{sel0}_s{sd}', f'{a0["what"]} — повтор (seed {sd})', arch=a0['arch'], pretrained=a0.get('pretrained', True),
             aug=a0.get('aug', True), size=a0.get('size', 256), seed=sd, epochs=a0.get('epochs', 15), bs=32 if 'b3' in a0['arch'] else 64,
             cw=a0.get('class_weights', False))
    reps = [R[k]['test'] for k in [sel0, f'{sel0}_s1', f'{sel0}_s2'] if k in R]
    R['selected_seed_summary'] = {m_: dict(mean=float(np.mean([r[m_] for r in reps])), std=float(np.std([r[m_] for r in reps], ddof=1)), n=len(reps))
                                  for m_ in ['accuracy', 'macro_f1', 'tumor_sens', 'tumor_spec', 'glioma_recall']} if len(reps) >= 2 else None
    save_R(); print('Повторы выбранного рецепта:', R['selected_seed_summary'])
else:
    print('Выбран ансамбль — повторы не нужны (он уже усредняет несколько обучений).')

In [ ]:
# === F1. ФИНАЛЬНАЯ ОЦЕНКА ВЫБРАННОЙ МОДЕЛИ ===
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, roc_auc_score, confusion_matrix
R = json.load(open(W/'experiments_v1.json')); PROB = W/'exp_probs'
sel = R['selected']['tag']; members = R[sel].get('members', [sel])
print('Выбрана по валидации:', sel, '| состав:', members, '| val macro-F1', round(R['selected']['val_macro_f1'], 4))
Z = [np.load(PROB/f'{k}.npz') for k in members]
Pva, Pte = np.mean([z['va'] for z in Z], 0), np.mean([z['te'] for z in Z], 0)
assert len(Pte) == len(TE) and len(Pva) == len(VA), 'Размеры не совпадают с разбиением'

def evaluate(P, Y):
    pr = P.argmax(1); tt = Y != 3; pt = 1 - P[:, 3]
    return dict(n=int(len(Y)), accuracy=float(accuracy_score(Y, pr)), macro_f1=float(f1_score(Y, pr, average='macro')),
                balanced_acc=float(balanced_accuracy_score(Y, pr)),
                recall={CLASSES[k]: (float((pr[Y == k] == k).mean()) if (Y == k).any() else None) for k in range(4)},
                tumor_sens=float(((pr != 3) & tt).sum() / max(tt.sum(), 1)), tumor_spec=float(((pr == 3) & ~tt).sum() / max((~tt).sum(), 1)),
                tumor_auc=(float(roc_auc_score(tt, pt)) if 0 < tt.sum() < len(Y) else None))
def boot(fn, groups, n=1000, seed=0):   # тот же бутстрэп по группам, что в основной работе
    rs2 = np.random.RandomState(seed); ug = np.unique(groups); by = {g: np.where(groups == g)[0] for g in ug}; vals = []
    for _ in range(n):
        ii = np.concatenate([by[g] for g in rs2.choice(ug, len(ug), replace=True)]); vals.append(fn(ii))
    return [round(float(x), 4) for x in np.percentile(vals, [2.5, 97.5])]

Y = LAB[TE]; pr = Pte.argmax(1); gT = df.group.values[TE]
F = {'selected': sel, 'members': members, 'val_macro_f1': R['selected']['val_macro_f1'], 'test_group': evaluate(Pte, Y)}
F['ci95'] = dict(accuracy=boot(lambda ii: (pr[ii] == Y[ii]).mean(), gT),
                 macro_f1=boot(lambda ii: f1_score(Y[ii], pr[ii], average='macro'), gT),
                 tumor_sens=boot(lambda ii: ((pr[ii] != 3) & (Y[ii] != 3)).sum() / max((Y[ii] != 3).sum(), 1), gT),
                 tumor_spec=boot(lambda ii: ((pr[ii] == 3) & (Y[ii] == 3)).sum() / max((Y[ii] == 3).sum(), 1), gT))
pos = {i: k for k, i in enumerate(TE)}
F['test_group_unique'] = evaluate(Pte[[pos[i] for i in TEU]], LAB[TEU])
fp_ = df.pid.values[TE] != ''
F['figshare_new_patients'] = evaluate(Pte[fp_], Y[fp_]); F['other_subset'] = evaluate(Pte[~fp_], Y[~fp_])
# режим скрининга — тот же способ выбора порога, что в основной работе (по валидации, не выше 0.5)
pv = 1 - Pva[:, 3]; thr = float(min(0.5, np.quantile(pv[LAB[VA] != 3], 0.01)))
pt_ = 1 - Pte[:, 3]; tt_ = Y != 3; fl = pt_ >= thr
F['screening'] = dict(thr=thr, sens=float((fl & tt_).sum() / tt_.sum()), spec=float((~fl & ~tt_).sum() / (~tt_).sum()))
t_ = df.iloc[TE].copy(); t_['ok'] = pr == Y
t_['res'] = pd.cut(t_[['h', 'w']].max(1), [0, 256, 400, 100000], labels=['до 256', '257-400', 'больше 400'])
F['by_resolution'] = {str(k): dict(acc=float(v['mean']), n=int(v['size'])) for k, v in t_.groupby('res', observed=True).ok.agg(['mean', 'size']).iterrows()}
F['confusion_matrix'] = confusion_matrix(Y, pr, labels=range(4)).tolist()
F['base_for_comparison'] = R['base']['test']
F['selected_seed_summary'] = R.get('selected_seed_summary')
json.dump(F, open(W/'results_final.json', 'w'), ensure_ascii=False, indent=1)
print(json.dumps({k: v for k, v in F.items() if k not in ('base_for_comparison',)}, ensure_ascii=False, indent=1))

cm = np.array(F['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5.2, 4.4)); ax.imshow(cm, cmap='Blues')
for i in range(4):
    for j in range(4): ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xticks(range(4), CLASSES, rotation=25); ax.set_yticks(range(4), CLASSES)
ax.set_xlabel('Предсказание модели'); ax.set_ylabel('Истина'); plt.tight_layout()
plt.savefig(W/'figs'/'cm_final.png', dpi=150); plt.show()

In [ ]:
# === F2. ВЕСА ВЫБРАННОЙ МОДЕЛИ ДЛЯ ДЕМО (части по 20 МБ — для загрузки на GitHub) ===
import hashlib
FD = W/'final'; shutil.rmtree(FD, ignore_errors=True); FD.mkdir()
manifest = []
for k in members:
    src_ = W/'ckpt'/('cls_v3_best.pt' if k == 'base' else f'exp_{k}_best.pt')
    arch = R[k].get('arch', 'efficientnet_b0'); size = R[k].get('size', 256)
    data = open(src_, 'rb').read(); parts = []
    for i in range(0, len(data), 20 * 1024 * 1024):
        nm = f'cls_{k}.pt.part{i // (20 * 1024 * 1024)}'; open(FD/nm, 'wb').write(data[i:i + 20 * 1024 * 1024]); parts.append(nm)
    manifest.append(dict(tag=k, arch=arch, size=size, parts=parts, sha256=hashlib.sha256(data).hexdigest()))
    print(f'{k}: {arch}, {len(data) / 1e6:.1f} МБ -> {len(parts)} частей')
json.dump(dict(selected=sel, models=manifest, screen_thr=F['screening']['thr']), open(FD/'cls_models.json', 'w'), indent=1)
print('Готово: папка almazov/final в Google Drive. Скачай её целиком (правый клик -> Скачать) в папку MRI_Downloader и напиши мне.')